In [ ]:
#dataset = "https://www.kaggle.com/datasets/puneet6060/intel-image-classification"

In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:
import zipfile
import os

# Define your zip file path and destination directory
zip_file = '/content/drive/MyDrive/dataset.zip'
destination_folder = '/content/dataset'

# Create destination folder if it doesn't exist
os.makedirs(destination_folder, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(destination_folder)

print("Dataset unzipped successfully.")


Dataset unzipped successfully.


In [5]:
!pip install torch torchvision lightning optuna wandb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from PIL import Image
import lightning as L
from lightning.pytorch import Trainer
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import ModelCheckpoint
import optuna
import wandb
from google.colab import drive

# =========================
# MOUNT GOOGLE DRIVE - Make sure drive is mounted
drive.mount('/content/drive')

# =========================
# WANDB SETUP - with error handling
try:
    wandb.login()
    WANDB_PROJECT = "transfer_learning_resnet_optuna_colab"
    use_wandb = True
except Exception as e:
    print(f"WandB login failed: {e}")
    print("Continuing without WandB logging")
    use_wandb = False

# =========================
# DATA PATHS
DATA_DIR = "/content/dataset/dataset"
TRAIN_DIR = os.path.join(DATA_DIR, "seg_train")
VAL_DIR = os.path.join(DATA_DIR, "seg_test")
PRED_DIR = os.path.join(DATA_DIR, "seg_pred")
CHECKPOINT_DIR = "/content/drive/MyDrive/checkpoints"  # Checkpoints directory in Google Drive

# Check if checkpoint directory exists, if not create it
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoint directory at {CHECKPOINT_DIR}")

# =========================
# HYPERPARAMETERS
BATCH_SIZE = 64
NUM_CLASSES = 6
IMAGE_SIZE = 224
N_TRIALS = 10
N_EPOCHS_TRIAL = 10
N_EPOCHS_FINAL = 20

idx_to_class = {0: 'buildings', 1: 'forest', 2: 'glacier', 3: 'mountain', 4: 'sea', 5: 'street'}

# =========================
# TRANSFORMS
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),  # Added rotation augmentation
    transforms.ColorJitter(brightness=0.1, contrast=0.1),  # Added color augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
])

# =========================
# DATALOADERS
def get_dataloaders():
    train_dataset = ImageFolder(TRAIN_DIR, transform=train_transforms)
    val_dataset = ImageFolder(VAL_DIR, transform=val_transforms)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print(f"Training dataset size: {len(train_dataset)}")
    print(f"Validation dataset size: {len(val_dataset)}")
    print(f"Class mapping: {train_dataset.class_to_idx}")

    return train_loader, val_loader

# =========================
# MODEL - Updated with correct validation hooks
class ResNetLightning(L.LightningModule):
    def __init__(self, backbone_name='resnet18', learning_rate=1e-3):
        super().__init__()
        self.save_hyperparameters()

        # Pretrained Model
        if backbone_name == 'resnet18':
            self.model = models.resnet18(weights='IMAGENET1K_V1')
        elif backbone_name == 'resnet34':
            self.model = models.resnet34(weights='IMAGENET1K_V1')
        else:
            raise ValueError("Backbone must be 'resnet18' or 'resnet34'")

        # Freeze ALL layers first
        for param in self.model.parameters():
            param.requires_grad = False

        # Fine-Tune layer4 and fc
        for name, child in self.model.named_children():
            if name == 'layer4':  # Fixed string literal comparison
                for param in child.parameters():
                    param.requires_grad = True

        # Replace final layer
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, NUM_CLASSES)

        # Always ensure FC layer is trainable
        for param in self.model.fc.parameters():
            param.requires_grad = True

        # Print model summary for debugging
        print(f"Using {backbone_name} backbone with learning rate {learning_rate}")
        print(f"Trainable parameters: {sum(p.numel() for p in self.parameters() if p.requires_grad)}")

        # Initialize validation metrics
        self.val_loss_list = []
        self.val_acc_list = []

    def forward(self, x):
        return self.model(x)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.parameters()),
            lr=self.hparams.learning_rate
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=3, verbose=True
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss",
                "interval": "epoch",
                "frequency": 1
            }
        }

    def training_step(self, batch, batch_idx):
        x, y = batch
        preds = self(x)
        loss = F.cross_entropy(preds, y)
        acc = (preds.argmax(dim=1) == y).float().mean()
        self.log('train_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('train_acc', acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        preds = self(x)
        loss = F.cross_entropy(preds, y)
        acc = (preds.argmax(dim=1) == y).float().mean()

        # Log metrics
        self.log('val_loss', loss, prog_bar=True, on_epoch=True)
        self.log('val_acc', acc, prog_bar=True, on_epoch=True)

        # Store for epoch end processing
        self.val_loss_list.append(loss)
        self.val_acc_list.append(acc)

        return {"val_loss": loss, "val_acc": acc}

    def on_validation_epoch_end(self):
        # Calculate average metrics
        avg_loss = torch.stack(self.val_loss_list).mean()
        avg_acc = torch.stack(self.val_acc_list).mean()

        # Print metrics
        print(f"Validation Epoch End: loss={avg_loss:.4f}, acc={avg_acc:.4f}")

        # Clear storage
        self.val_loss_list = []
        self.val_acc_list = []

# =========================
# PREDICTION DATASET
class PredictDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, transform):
        self.root_dir = root_dir
        self.transform = transform
        # Filter for image files only
        valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp'}
        self.image_paths = [
            os.path.join(root_dir, fname) for fname in os.listdir(root_dir)
            if os.path.splitext(fname.lower())[1] in valid_extensions
        ]
        print(f"Found {len(self.image_paths)} images for prediction")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert("RGB")
            image = self.transform(image)
            return image, img_path
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")
            # Return a blank image and the path
            blank_image = torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)
            return blank_image, img_path

# =========================
# OPTUNA OBJECTIVE
def objective(trial):
    backbone_name = trial.suggest_categorical("backbone", ["resnet18", "resnet34"])
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    print(f"\nStarting Trial {trial.number}")

    # Create model
    model = ResNetLightning(backbone_name=backbone_name, learning_rate=learning_rate)

    # Setup logger
    if use_wandb:
        wandb_logger = WandbLogger(project=WANDB_PROJECT, name=f"trial_{trial.number}")
        logger = wandb_logger
    else:
        logger = True  # Use default Lightning logger

    # Create checkpoint callback for trials
    checkpoint_callback = ModelCheckpoint(
        monitor="val_loss",
        dirpath=os.path.join(CHECKPOINT_DIR, f"trial_{trial.number}"),
        filename="best_model",
        save_top_k=1,
        mode="min",
    )

    # Create data loaders
    train_loader, val_loader = get_dataloaders()

    # Create the trainer
    trainer = Trainer(
        max_epochs=N_EPOCHS_TRIAL,
        logger=logger,
        deterministic=True,
        callbacks=[checkpoint_callback],
        accelerator="auto",  # Lightning will automatically choose the device
        devices="auto"       # Let Lightning handle device selection
    )

    # Train the model
    trainer.fit(model, train_loader, val_loader)

    # Get the validation loss from the last epoch
    val_loss = trainer.callback_metrics["val_loss"].item()
    print(f"Trial {trial.number} completed — Validation Loss: {val_loss:.4f}")
    return val_loss


# =========================
# FINAL TRAIN FUNCTION
def train_final_model(best_params):
    print("\nTraining Final Model with Best Hyperparameters:")
    print(f"Backbone: {best_params['backbone']}")
    print(f"Learning Rate: {best_params['learning_rate']:.6f}")

    # Create model with best params
    model = ResNetLightning(
        backbone_name=best_params['backbone'],
        learning_rate=best_params['learning_rate']
    )

    # Setup logger
    if use_wandb:
        wandb_logger = WandbLogger(project=WANDB_PROJECT, name="final_training")
        logger = wandb_logger
    else:
        logger = True  # Use default Lightning logger

    # Create dataloaders
    train_loader, val_loader = get_dataloaders()

    # Create checkpoint callback for final training
    checkpoint_callback = ModelCheckpoint(
        monitor="val_loss",
        dirpath=CHECKPOINT_DIR,
        filename="best_model",
        save_top_k=1,
        mode="min",
        verbose=True
    )

    # Check for existing final model checkpoint
    final_checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_model.ckpt")
    ckpt_path = None

    if os.path.exists(final_checkpoint_path):
        print(f"Found existing checkpoint at {final_checkpoint_path}")
        ckpt_path = final_checkpoint_path
    else:
        print("No existing checkpoint found, starting from scratch")

    # Create trainer
    trainer = Trainer(
        max_epochs=N_EPOCHS_FINAL,
        logger=logger,
        accelerator="auto",
        devices="auto",
        callbacks=[checkpoint_callback]
    )

    # Start or resume training
    # Note: We pass ckpt_path to fit method, not to Trainer constructor
    trainer.fit(model, train_loader, val_loader, ckpt_path=ckpt_path)

    # Save the final model - separate from best model checkpoint
    final_model_path = os.path.join(CHECKPOINT_DIR, "final_model.ckpt")
    trainer.save_checkpoint(final_model_path)
    print(f"Final model saved to {final_model_path}")

    # Log model as artifact to Wandb if available
    if use_wandb:
        try:
            best_model_artifact = wandb.Artifact("best_model", type="model")
            best_model_path = checkpoint_callback.best_model_path
            if best_model_path:
                best_model_artifact.add_file(best_model_path)
                wandb.log_artifact(best_model_artifact)
                print(f"Logged best model artifact to WandB: {best_model_path}")
        except Exception as e:
            print(f"Error logging model artifact to WandB: {e}")

    # Return the best model based on validation performance
    if checkpoint_callback.best_model_path:
        # Load the best model
        print(f"Loading best model from {checkpoint_callback.best_model_path}")
        best_model = ResNetLightning.load_from_checkpoint(checkpoint_callback.best_model_path)
        return best_model
    else:
        # Return the final model
        return model

# =========================
# PREDICTION FUNCTION
def predict(model):
    # Put model in evaluation mode
    model.eval()
    model.freeze()

    # Create prediction dataset and loader
    pred_dataset = PredictDataset(PRED_DIR, val_transforms)
    pred_loader = DataLoader(pred_dataset, batch_size=32, shuffle=False)

    # Check if prediction dataset is empty
    if len(pred_dataset) == 0:
        print(f"No images found in {PRED_DIR} for prediction!")
        return

    # Lists to store predictions and paths
    preds = []
    paths = []

    # Get device
    device = next(model.parameters()).device
    print(f"Running predictions on device: {device}")

    # Run predictions
    with torch.no_grad():
        for batch, paths_batch in pred_loader:
            batch = batch.to(device)
            outputs = model(batch)
            predicted_classes = torch.argmax(outputs, dim=1)
            preds.extend(predicted_classes.cpu().numpy())
            paths.extend(paths_batch)

    # Save predictions
    predictions_path = os.path.join(CHECKPOINT_DIR, "predictions.txt")
    with open(predictions_path, "w") as f:
        for path, pred in zip(paths, preds):
            class_name = idx_to_class[pred]
            f.write(f"{os.path.basename(path)} --> {class_name}\n")

    print(f"Predictions saved to {predictions_path}")

# =========================
# MAIN FUNCTION
def main():
    print("Starting the training pipeline...")

    # Check if dataset exists
    if not os.path.exists(TRAIN_DIR) or not os.path.exists(VAL_DIR):
        print(f"Dataset not found at {DATA_DIR}")
        print("Please make sure the dataset is properly extracted and structured.")
        return

    # Print dataset structure
    print(f"Train directory: {TRAIN_DIR}")
    print(f"Validation directory: {VAL_DIR}")
    print(f"Prediction directory: {PRED_DIR}")

    print("\nStarting hyperparameter optimization with Optuna...")

    # OPTUNA OPTIMIZATION
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=N_TRIALS)

    # Get the best trial
    best_trial = study.best_trial
    best_params = best_trial.params
    print(f"\nBest Trial: {best_trial.number}")
    print(f"Best Params: {best_params}")
    print(f"Best validation loss: {best_trial.value:.4f}")

    # Train the final model with best hyperparameters
    print("\nStarting final model training...")
    trained_model = train_final_model(best_params)

    # Check if prediction directory exists
    if os.path.exists(PRED_DIR):
        print(f"\nMaking predictions on images in {PRED_DIR}")
        # Make predictions using the trained model
        predict(trained_model)
    else:
        print(f"\nPrediction directory {PRED_DIR} not found. Skipping prediction step.")

    print("\nTraining pipeline completed successfully!")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


...

INFO: Epoch 18, global step 4180: 'val_loss' was not in top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 18, global step 4180: 'val_loss' was not in top 1


Validation Epoch End: loss=0.1898, acc=0.9309


Validation: |          | 0/? [00:00<?, ?it/s]

INFO: Epoch 19, global step 4400: 'val_loss' was not in top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 19, global step 4400: 'val_loss' was not in top 1
INFO: `Trainer.fit` stopped: `max_epochs=20` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.


Validation Epoch End: loss=0.1883, acc=0.9340
Final model saved to /content/drive/MyDrive/checkpoints/final_model.ckpt
Logged best model artifact to WandB: /content/drive/MyDrive/checkpoints/best_model.ckpt
Loading best model from /content/drive/MyDrive/checkpoints/best_model.ckpt
Using resnet18 backbone with learning rate 1.0172746685092487e-05
Trainable parameters: 8396806

Making predictions on images in /content/dataset/dataset/seg_pred
Found 7301 images for prediction
Running predictions on device: cuda:0
Predictions saved to /content/drive/MyDrive/checkpoints/predictions.txt

Training pipeline completed successfully!
